# Data Consumption Architecture

How to share `training.gold` data with Databricks, Snowflake, Power BI, tools and apps. Examples first, comparison tables at the end.

## Example 1: Databricks to Databricks (Delta Sharing)
```sql
-- Provider
CREATE SHARE IF NOT EXISTS customer_share;
ALTER SHARE customer_share ADD TABLE training.gold.customers WITH HISTORY;
CREATE RECIPIENT partner_dbx USING ID 'azure:westeurope:<consumer-metastore-id>';
GRANT SELECT ON SHARE customer_share TO RECIPIENT partner_dbx;

-- Consumer (other Databricks account)
CREATE CATALOG partner_customers USING SHARE <provider_name>.customer_share;
SELECT * FROM partner_customers.gold.customers;
```

## Example 2: Open sharing (Snowflake, Power BI, pandas, Spark)
```sql
CREATE RECIPIENT external_team;                 -- token-based recipient
GRANT SELECT ON SHARE customer_share TO RECIPIENT external_team;
DESCRIBE RECIPIENT external_team;               -- activation_link -> consumer downloads config.share
```
```python
# Consumer side, any Python environment
import delta_sharing
df = delta_sharing.load_as_pandas("config.share#customer_share.gold.customers")
```

## Example 3: Snowflake via Iceberg REST catalog
```sql
-- Databricks: make the Delta table readable as Iceberg
ALTER TABLE training.gold.customers SET TBLPROPERTIES (
  'delta.enableIcebergCompatV2' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg');
GRANT USE CATALOG ON CATALOG training TO `<sp-app-id>`;
GRANT USE SCHEMA, SELECT, EXTERNAL USE SCHEMA ON SCHEMA training.gold TO `<sp-app-id>`;
```
```sql
-- Snowflake
CREATE CATALOG INTEGRATION databricks_uc
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  CATALOG_NAMESPACE = 'gold'
  REST_CONFIG = (
    CATALOG_URI = 'https://<workspace-host>/api/2.1/unity-catalog/iceberg-rest'
    WAREHOUSE = 'training'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS)
  REST_AUTHENTICATION = (
    TYPE = OAUTH
    OAUTH_TOKEN_URI = 'https://<workspace-host>/oidc/v1/token'
    OAUTH_CLIENT_ID = '<sp-app-id>'
    OAUTH_CLIENT_SECRET = '<sp-secret>'
    OAUTH_ALLOWED_SCOPES = ('all-apis'))
  ENABLED = TRUE;

CREATE ICEBERG TABLE customers
  CATALOG = 'databricks_uc'
  CATALOG_TABLE_NAME = 'customers';
```

## Example 4: Power BI (SQL warehouse)
Power BI Desktop > Get data > **Azure Databricks**
* Server hostname: `<workspace-host>`
* HTTP path: `/sql/1.0/warehouses/<warehouse-id>`
* Auth: Microsoft Entra ID (users) or service principal (scheduled refresh)
* Mode: **DirectQuery** for live / large data, **Import** for small, fast dashboards

Or Catalog Explorer > table > **Use with BI tools > Publish to Power BI workspace**.
```sql
GRANT USE CATALOG ON CATALOG training TO `business_users`;
GRANT USE SCHEMA, SELECT ON SCHEMA training.gold TO `business_users`;
-- + CAN USE on the SQL warehouse
```

## Example 5: JDBC (Java, Tableau, other tools)
```text
jdbc:databricks://<workspace-host>:443/default;transportMode=http;ssl=1;
httpPath=/sql/1.0/warehouses/<warehouse-id>;AuthMech=11;Auth_Flow=1;
OAuth2ClientId=<sp-app-id>;OAuth2Secret=<sp-secret>
```

## Example 6: Apps (Python SQL connector / REST API)
```python
from databricks import sql
with sql.connect(server_hostname="<workspace-host>",
                 http_path="/sql/1.0/warehouses/<warehouse-id>",
                 access_token="<token>") as conn:
    rows = conn.cursor().execute("SELECT * FROM training.gold.customers LIMIT 10").fetchall()
```
```bash
curl -X POST https://<workspace-host>/api/2.0/sql/statements \
  -H "Authorization: Bearer <token>" \
  -d '{"warehouse_id":"<warehouse-id>","statement":"SELECT * FROM training.gold.customers LIMIT 10"}'
```

## Privileges to share
```sql
GRANT CREATE SHARE, CREATE RECIPIENT ON METASTORE TO `admin@one-consultants.com`;
-- Share owner also needs SELECT on every table added to the share
```

## Which mechanism for which consumer

| Consumer | Mechanism | Copy? | Compute paid by |
|---|---|---|---|
| Databricks user, **same** account + metastore | Unity Catalog `GRANT` | no | consumer |
| Databricks user, **other** account / region / cloud | Delta Sharing (Databricks-to-Databricks) | no | consumer |
| Snowflake | Iceberg REST catalog (UniForm) or Delta Sharing (open) | no | consumer |
| Power BI, interactive / DirectQuery / per-user security | Power BI connector to a SQL warehouse (ODBC / JDBC) | no | provider (warehouse) |
| Power BI, scheduled import, external org | Power BI Delta Sharing connector | import | consumer |
| Tableau, Excel, Java / .NET tools | JDBC / ODBC to a SQL warehouse | no | provider |
| Custom apps, APIs | SQL Statement Execution API, Python SQL connector | no | provider |
| Spark / Trino / DuckDB outside Databricks | Iceberg REST catalog or Delta Sharing (open) | no | consumer |
| Apps needing millisecond lookups | Lakebase synced table (Postgres) | yes | provider |
| Public / commercial data products | Databricks Marketplace (Delta Sharing) | no | consumer |
| Joins with a partner without exposing raw data | Clean Rooms | no | both |
| Partner wants files | Export CSV / Parquet to a volume or external location | yes | provider |

## Delta Sharing vs JDBC / ODBC

| | Delta Sharing | JDBC / ODBC |
|---|---|---|
| Consumer needs | own compute | only a driver + network access |
| Provider cost | none (storage egress only) | SQL warehouse running |
| Security | share-level (tables, partitions, views) | per-user UC grants, row filters, column masks |
| Access | read only | read (and write if granted) |
| Freshness | live, no copy | live, no copy |